# 应许之路 · 0 成本满血 PBR 商业 3D 工坊（Microsoft TRELLIS · MIT 许可）

- **费用**：$0（Google Colab 免费 T4 16GB GPU）
- **商用许可**：Microsoft TRELLIS 官方权重采用 **MIT License**（完全免费商用、零版税、无用户数限制）。
- **一键产出（与 Godot 4.7.2 槽位 100% 同名）**：
  1. `CHR_Abram.glb`（4 视图多视角重建 + 1024px PBR）
  2. `BLD_MainTent.glb`（雪松木主帐 + 织锦旗 + 1024px PBR）
  3. `BLD_StoneWell.glb`（迦南圆石水井 + 辘轳木桶 + 1024px PBR）
  4. `CHR_Sheep.glb`（古近东肥尾绵羊 + 1024px PBR）
- **操作步骤**：顶部菜单 `代码执行程序 → 更改运行时类型 → T4 GPU`，然后点 `代码执行程序 → 全部运行` 即可自动打包下载 `promise_path_glb_pack.zip`。

In [ ]:
# 1. 检查 T4 GPU 并安装 Microsoft TRELLIS (MIT) 兼容环境（首次约 3 分钟）
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS || true
%cd /content/TRELLIS
!pip install -q pillow imageio imageio-ffmpeg tqdm easydict opencv-python-headless scipy ninja rembg onnxruntime trimesh xatlas pyvista pymeshfix igraph transformers
!pip install -q git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8
!pip install -q nvdiffrast || pip install -q git+https://github.com/NVlabs/nvdiffrast.git
!pip install -q spconv-cu121 || pip install -q spconv-cu120
!pip install -q xformers || true
print('TRELLIS_COLAB_ENV_READY')

In [ ]:
# 2. 准备输入素材包：优先从 GitHub 仓库自动拉取；若为私有仓库则自动弹出本地上传按钮（上传 colab_inputs_pack.zip 即可）
import os, zipfile, urllib.request
IN_DIR = '/content/inputs'
os.makedirs(IN_DIR, exist_ok=True)
ZIP_PATH = '/content/colab_inputs_pack.zip'
RAW_URL = 'https://github.com/Paul30028/promise-path-next/raw/main/design/review-slice-01/colab_inputs_pack.zip'

downloaded = False
try:
    urllib.request.urlretrieve(RAW_URL, ZIP_PATH)
    if os.path.exists(ZIP_PATH) and zipfile.is_zipfile(ZIP_PATH):
        downloaded = True
        print('已自动从 GitHub 拉取素材包!')
except Exception as e:
    print('自动拉取未命中（可能是私有仓库），转为手动上传:', e)

if not downloaded:
    from google.colab import files
    print('请选择本地 c:\\AI\\gemini\\promise-path-next\\design\\review-slice-01\\colab_inputs_pack.zip')
    up = files.upload()
    for fn in up.keys():
        if fn.endswith('.zip'):
            ZIP_PATH = os.path.abspath(fn)
        else:
            os.replace(fn, os.path.join(IN_DIR, os.path.basename(fn)))

if os.path.exists(ZIP_PATH) and zipfile.is_zipfile(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(IN_DIR)

print('就绪底图:', sorted(os.listdir(IN_DIR)))

In [ ]:
# 3. 满血批量生成 4 个 1024px PBR 商业 .glb 并自动打包下载 promise_path_glb_pack.zip
import os, gc, zipfile, torch
os.environ['SPCONV_ALGO'] = 'native'
os.environ['ATTN_BACKEND'] = 'sdpa'  # T4 16GB 最稳零报错后端
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils
from google.colab import files

IN_DIR = '/content/inputs'
OUT_DIR = '/content/glb_out'
os.makedirs(OUT_DIR, exist_ok=True)

pipeline = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')
pipeline.cuda()

def export_glb(outputs, name, simplify=0.92, tex_size=1024):
    glb = postprocessing_utils.to_glb(
        outputs['gaussian'][0],
        outputs['mesh'][0],
        simplify=simplify,
        texture_size=tex_size,
    )
    out_p = os.path.join(OUT_DIR, f'{name}.glb')
    glb.export(out_p)
    print(f'SUCCESS: {name}.glb ({os.path.getsize(out_p):,} bytes)')
    del outputs, glb
    gc.collect()
    torch.cuda.empty_cache()

# A. 英雄角色：亚伯兰（4 视图多视角满血重建！）
abram_views = [os.path.join(IN_DIR, f) for f in ['abram_front.png', 'abram_back.png', 'abram_left.png', 'abram_right_flipped.png'] if os.path.exists(os.path.join(IN_DIR, f))]
if len(abram_views) >= 2:
    print(f'=== Generating CHR_Abram.glb ({len(abram_views)} views) ===')
    imgs = [Image.open(p).convert('RGBA') for p in abram_views]
    out = pipeline.run_multi_image(imgs, seed=1234)
    export_glb(out, 'CHR_Abram', simplify=0.92, tex_size=1024)
elif len(abram_views) == 1:
    out = pipeline.run(Image.open(abram_views[0]).convert('RGBA'), seed=1234)
    export_glb(out, 'CHR_Abram', simplify=0.92, tex_size=1024)

# B. 建筑与生物：主帐、石井、肥尾绵羊
for name, fname, seed in [
    ('BLD_MainTent', 'BLD_MainTent.png', 2026),
    ('BLD_StoneWell', 'BLD_StoneWell.png', 2027),
    ('CHR_Sheep', 'CHR_Sheep.png', 2028),
]:
    p = os.path.join(IN_DIR, fname)
    if os.path.exists(p):
        print(f'=== Generating {name}.glb ===')
        out = pipeline.run(Image.open(p).convert('RGBA'), seed=seed)
        export_glb(out, name, simplify=0.92, tex_size=1024)

PACK_ZIP = '/content/promise_path_glb_pack.zip'
with zipfile.ZipFile(PACK_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(OUT_DIR)):
        if fn.endswith('.glb'):
            zf.write(os.path.join(OUT_DIR, fn), fn)
print('ALL DONE ->', PACK_ZIP, os.path.getsize(PACK_ZIP), 'bytes')
files.download(PACK_ZIP)